# 03 · Digital filters

Filters appear everywhere in a transceiver: anti-aliasing before a
decimation, the electrical bandwidth of a photodiode or a DAC, channel
selection, noise limiting.  CommKit keeps **design** and **application**
apart: a design function returns coefficients (`fir_taps`,
`butterworth_sos`, ...), and an apply function uses them (`fir_filter`,
`iir_filter`).

1. FIR design by the window method: length against transition width;
2. applying an FIR filter, aligned or causal;
3. the IIR families and their trade-off between selectivity and phase;
4. causal against zero-phase filtering;
5. what *smoothing* is for, and why it is not in the signal chain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    return to_device(a, "cpu")

## 1. FIR filters by the window method

An ideal low-pass filter has a sinc impulse response of infinite length.
The window method truncates it to `num_taps` samples and tapers the edges
with a window.  The result is **linear phase** (a symmetric impulse
response), so every frequency is delayed by the same `(num_taps - 1) / 2`
samples, and the transition band narrows as `1 / num_taps`.
`plot_filter_response` shows the impulse, magnitude, phase and group delay
of any design.  `causal=True` draws the filter as the causal convolution
computes it, so the delay of `(101 - 1) / 2 = 50` samples, 50 µs at 1 MS/s,
shows in the phase and the group delay; section 2 explains the default.
`annotations=True` adds a dashed line 3 dB below the peak of the magnitude.

In [ ]:
fs = 1e6
taps = ck.filtering.fir_taps(sampling_rate=fs, num_taps=101, cutoff=100e3)
ck.plotting.plot_filter_response(
    taps, sampling_rate=fs, causal=True, annotations=True, show=True
)

ax = None
for num_taps in (21, 51, 201):
    taps = ck.filtering.fir_taps(sampling_rate=fs, num_taps=num_taps, cutoff=100e3)
    _, ax = ck.plotting.plot_filter_response(
        taps,
        sampling_rate=fs,
        nfft=4096,
        quantity="magnitude",
        relative=True,
        annotations=True,
        ax=ax,
        label=f"{num_taps} taps",
        ylim=(-90, 5),
        title="Hamming-window low-pass, cutoff 100 kHz",
        legend=True,
    )
    response = ck.filtering.filter_response(taps, sampling_rate=fs, nfft=1 << 15)
    level = 20 * np.log10(np.abs(host(response.response)))
    f = host(response.frequencies)
    print(
        f"{num_taps:3d} taps: -6 dB at {f[np.argmax(level < -6.02)] / 1e3:5.1f} kHz, "
        f"-3 dB at {f[np.argmax(level < -3)] / 1e3:5.1f} kHz"
    )
plt.show()

The window method puts `cutoff` at **half amplitude**, -6 dB, whatever the
length: all three designs cross -6 dB within 0.3 kHz of 100 kHz.  The -3 dB line
(`relative=True` draws each response relative to its peak, 0 dB, so the
overlays share one line) is crossed earlier, and the shorter the filter,
the earlier: at 80 kHz for 21 taps, 98 kHz for 201.  Compare filters by the
edge that matters to the application, not by the `cutoff` argument.

## 2. Applying a FIR filter

`fir_filter` convolves along the time axis and keeps the output aligned
with the input (the filter delay is removed), so sample `n` of the output
belongs to sample `n` of the input; `causal=True` gives the plain causal
convolution instead.  It picks the fastest method from the sizes: direct
convolution for short filters, one FFT of the record, or overlap-save FFT
blocks for long records (chromatic-dispersion compensation needs thousands
of taps), all giving the same samples.

Here a 2 MBd QPSK signal sits next to a strong interferer at 3.5 MHz; a
low-pass filter removes it.

In [ ]:
tx = ck.generate(
    Constellation.psk(4),
    num_symbols=2**14,
    symbol_rate=2e6,
    sps=8,
    pulse=ck.RRC(rolloff=0.25),
    rng=1,
    device=DEVICE,
)
x = ck.spectral.add_pilot_tone(tx, frequency=3.5e6, power_ratio_db=10)

lp = ck.filtering.fir_taps(sampling_rate=tx.sampling_rate, num_taps=129, cutoff=1.6e6)
y_aligned = ck.filtering.fir_filter(x, lp)
y_causal = ck.filtering.fir_filter(x, lp, causal=True)
delay = (lp.size - 1) // 2
err = np.max(np.abs(host(y_aligned.samples)[:-delay] - host(y_causal.samples)[delay:]))
print(
    f"causal output = aligned output {delay} samples later (max difference {err:.1e})"
)

_, ax = ck.plotting.plot_psd(x, nperseg=1024, title=None, label="with interferer")
ck.plotting.plot_psd(
    y_aligned,
    nperseg=1024,
    ax=ax,
    title="129-tap low-pass at 1.6 MHz",
    label="filtered",
    legend=True,
    show=True,
)

In time, the interferer is a fast ripple 10 dB stronger than the QPSK
waveform, and it hides the symbols entirely.  After the filter the
waveform is the transmitted one again, with no delay: `fir_filter`
compensates the `(129 - 1) / 2` samples of the linear-phase filter.

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1)), sharey=True
)
ck.plotting.plot_time_domain(x, num_symbols=10, ax=axes[0], title="With interferer")
ck.plotting.plot_time_domain(
    y_aligned, num_symbols=10, ax=axes[1], title=None, label="filtered"
)
ck.plotting.plot_time_domain(
    tx, num_symbols=10, ax=axes[1], title="Filtered", label="transmitted", legend=True
)
plt.show()

`plot_filter_response` draws an FIR filter as `fir_filter` applies it.  By
default, aligned on the centre tap, this linear-phase filter has zero
group delay; with `causal=True` it is the causal convolution's 64
samples, 4 µs at 16 MS/s.  The dashboard of section 1 used
`causal=True` for that reason.  IIR filters are causal either way.

In [ ]:
_, ax = ck.plotting.plot_filter_response(
    lp,
    sampling_rate=tx.sampling_rate,
    quantity="group_delay",
    label="fir_filter(x, lp)",
)
ck.plotting.plot_filter_response(
    lp,
    sampling_rate=tx.sampling_rate,
    quantity="group_delay",
    causal=True,
    ax=ax,
    label="fir_filter(x, lp, causal=True)",
    legend=True,
    xlim=(0, 1.6e6),
    ylim=(-1e-6, 5e-6),
    title="Group delay of the 129-tap low-pass",
    show=True,
)

## 3. IIR families

An IIR filter reaches a given selectivity with far fewer coefficients, at
the cost of a nonlinear phase.  CommKit designs them as cascades of
second-order sections (SOS, numerically robust).  At the same order the
classic families trade steepness against phase linearity:

| design | passband | stopband | group delay |
|---|---|---|---|
| `bessel_sos` | smooth, slow roll-off | slow | flattest (best pulse fidelity) |
| `butterworth_sos` | maximally flat | moderate | moderate peak at the edge |
| `chebyshev1_sos` | ripple | steep | larger peak |
| `chebyshev2_sos` | flat | ripple, steep | moderate |
| `elliptic_sos` | ripple | ripple, steepest | largest peak |

A Bessel response is the usual model of a receiver's electrical bandwidth
because its flat group delay keeps pulses undistorted.

In [ ]:
designs = {
    "Bessel": ck.filtering.bessel_sos(sampling_rate=fs, cutoff=100e3, order=6),
    "Butterworth": ck.filtering.butterworth_sos(
        sampling_rate=fs, cutoff=100e3, order=6
    ),
    "Chebyshev I": ck.filtering.chebyshev1_sos(
        sampling_rate=fs, cutoff=100e3, order=6, ripple=1.0
    ),
    "Chebyshev II": ck.filtering.chebyshev2_sos(
        sampling_rate=fs, cutoff=100e3, order=6, attenuation=50.0
    ),
    "Elliptic": ck.filtering.elliptic_sos(
        sampling_rate=fs, cutoff=100e3, order=6, ripple=1.0, attenuation=50.0
    ),
}
ax_m = ax_g = None
for name, sos in designs.items():
    _, ax_m = ck.plotting.plot_filter_response(
        sos,
        sampling_rate=fs,
        nfft=4096,
        quantity="magnitude",
        relative=True,
        annotations=True,
        ax=ax_m,
        label=name,
        ylim=(-80, 5),
        title="Order 6, cutoff 100 kHz",
        legend=True,
    )
    _, ax_g = ck.plotting.plot_filter_response(
        sos,
        sampling_rate=fs,
        nfft=4096,
        quantity="group_delay",
        ax=ax_g,
        label=name,
        xlim=(0, 100e3),
        title="Group delay across the passband",
        legend=True,
    )
plt.show()
for name, sos in designs.items():
    response = ck.filtering.filter_response(sos, sampling_rate=fs, nfft=1 << 15)
    level = 20 * np.log10(np.abs(host(response.response)))
    print(
        f"{name:13s} -3 dB at {host(response.frequencies)[np.argmax(level < -3)] / 1e3:6.1f} kHz"
    )

All five are designed with `cutoff=100e3`, and the -3 dB line shows that
the word means something different in each family.  Butterworth is -3 dB
at the cutoff.  Chebyshev I and elliptic put it at the edge of their 1 dB
passband ripple and reach -3 dB a little above.  Chebyshev II puts it at
the start of its 50 dB stopband, so -3 dB comes at 62 kHz.  Bessel's
default `norm="phase"` matches the phase of the analog prototype, -3 dB at
59 kHz; `norm="mag"` would put -3 dB at the cutoff.

The group-delay peak at the band edge shows up in the time domain as
overshoot and ringing.  The steep designs ring for many samples; the Bessel
step barely overshoots.  (Even-order Chebyshev I and elliptic designs also
settle at the bottom of their 1 dB passband ripple, a gain of 0.89 at DC.)

`step_response` computes the causal response to a unit step directly from each filter. The time axis starts at the step, so the ringing of the designs can be compared directly; `annotations=True` marks each final value with a dashed line.

In [ ]:
ax = None
for name, sos in designs.items():
    y = host(ck.filtering.step_response(sos, num_samples=200))
    print(
        f"{name:13s} overshoot {100 * (y.max() / y[-1] - 1):5.1f} %, final value {y[-1]:.3f}"
    )
    _, ax = ck.plotting.plot_filter_response(
        sos,
        sampling_rate=fs,
        num_impulse_samples=200,
        quantity="step",
        annotations=True,
        ax=ax,
        label=name,
        legend=True,
    )
plt.show()

## 4. Causal or zero-phase

`iir_filter(sig, sos)` runs the causal filter a real device implements,
with its frequency-dependent delay; use it to model a DAC, driver or
photodiode bandwidth.  `iir_filter(..., zero_phase=True)` filters forward
and then backward: the phase cancels, the magnitude response is squared,
and there is no delay.  This needs the whole record and models no real
device, but it is what offline analysis (smoothing, drift extraction)
wants.

In [ ]:
pulse = np.zeros(300, dtype=np.float32)
pulse[100:110] = 1.0
sos = designs["Butterworth"]
_, ax = ck.plotting.plot_time_domain(pulse, sampling_rate=fs, title=None, label="input")
ck.plotting.plot_time_domain(
    ck.filtering.iir_filter(pulse, sos),
    sampling_rate=fs,
    ax=ax,
    title=None,
    label="causal (delayed)",
)
ck.plotting.plot_time_domain(
    ck.filtering.iir_filter(pulse, sos, zero_phase=True),
    sampling_rate=fs,
    ax=ax,
    title="A 10 µs pulse through the Butterworth filter",
    label="zero-phase (centered)",
    xlim=(80e-6, 160e-6),
    legend=True,
    show=True,
)

## 5. Smoothing is for looking, not for the signal chain

`commkit.smoothing` (`moving_average`, `savgol_smooth`,
`smooth_density_2d`) smooths *derived* quantities for display or
estimation: an error curve, a phase trajectory, a histogram.  They have no
meaningful frequency response in a link budget, so they never sit in the
signal path; use the filters above for that.

In [ ]:
rng = np.random.default_rng(2)
curve = np.exp(-np.arange(2000) / 400) + 0.05 * rng.standard_normal(2000)
fig, ax = plt.subplots()
ax.plot(curve, alpha=0.4, label="raw (e.g. an equalizer's squared error)")
ax.plot(ck.smoothing.moving_average(curve, window=51), label="moving average, 51")
ax.plot(
    ck.smoothing.savgol_smooth(curve, window=101, polyorder=3),
    label="Savitzky-Golay, 101 / 3",
)
ax.legend()
plt.show()

**Summary**

* Design once, apply anywhere: `fir_taps` / `*_sos` return coefficients;
  `fir_filter` and `iir_filter` apply them on the data's device.
* FIR: linear phase, transition width `∝ 1/num_taps`; `fir_filter` picks
  direct, FFT or overlap-save convolution from the sizes.
* IIR: selectivity against group-delay flatness; Bessel for pulse fidelity.
* `zero_phase=True` for offline analysis, `False` to model a real device.
* `plot_filter_response` draws FIR filters as `fir_filter` applies them
  (`causal=` for the other reference); `relative=True` compares designs
  of any scaling, and `annotations=True` adds the -3 dB, symbol and
  final-value guides.

**Next:** [04 · Pulse shaping and matched filtering](04-pulse-shaping-and-matched-filtering.ipynb).